In [32]:
import os
import pandas as pd
import numpy as np

In [33]:
methods_names = {
    "dist_linguistic_confidence": "Dist. Ling. Conf.",
    "dist_semantic_uncertainty": "Dist. Semantic Unc.",
    "dist_lnll": "Dist. Token Prob"
}

dataset_size = {
    "mmlu": 116000,
    "squadv2": 130000,
    "truthful_qa": 817
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuAD2.0",
    "truthful_qa": "TruthfulQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen2.5-7B-Instruct": "Qwen2.5-7B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
}

In [34]:
prompt_type = "direct_qa"

In [35]:
results_dir = f"/hdd/ivny/{prompt_type}_cross_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    training_set, test_set = dataset_name.split("--")
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["model"] = model_name_map.get(model_name, model_name)
            record["training_set"] = dataset_map.get(training_set, training_set)
            record["test_set"] = dataset_map.get(test_set, test_set)
            record["dataset_size"] = dataset_size.get(test_set, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [63]:
df = pd.DataFrame(all_records).drop(columns=["model", "dataset_size"])
dataset_weighted_average_mean = df.groupby(["training_set", "test_set"]).mean().reset_index()

In [64]:
dataset_weighted_average_mean.columns.tolist()

['training_set',
 'test_set',
 'original_lc_generalised_ECE',
 'original_lc_faithfulness_divergence',
 'original_lc_ece_mean',
 'original_lc_dAUROC',
 'original_lc_auroc_mean',
 'original_tp_generalised_ECE',
 'original_tp_faithfulness_divergence',
 'original_tp_ece_mean',
 'original_tp_dAUROC',
 'original_tp_auroc_mean',
 'original_su_generalised_ECE',
 'original_su_faithfulness_divergence',
 'original_su_ece_mean',
 'original_su_dAUROC',
 'original_su_auroc_mean',
 'calibrated_lc_generalised_ECE',
 'calibrated_lc_faithfulness_divergence',
 'calibrated_lc_ece_mean',
 'calibrated_lc_dAUROC',
 'calibrated_lc_auroc_mean',
 'calibrated_tp_generalised_ECE',
 'calibrated_tp_faithfulness_divergence',
 'calibrated_tp_ece_mean',
 'calibrated_tp_dAUROC',
 'calibrated_tp_auroc_mean',
 'calibrated_su_generalised_ECE',
 'calibrated_su_faithfulness_divergence',
 'calibrated_su_ece_mean',
 'calibrated_su_dAUROC',
 'calibrated_su_auroc_mean',
 'calibrated_lc_rewritten_lc_generalised_ECE',
 'calibrate

In [77]:
fd_improvement_df = dataset_weighted_average_mean[["training_set", "test_set"]].copy()


fd_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"])
fd_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"])
fd_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"])

fd_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -1.6908431937964359,
  'Token Probability': -1.6666830271860136,
  'Semantic Uncertainty': -1.6318554554167854},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -1.3866581283549761,
  'Token Probability': -1.6250467984834072,
  'Semantic Uncertainty': -1.4102723858597999},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -1.2546781601482992,
  'Token Probability': -1.22705492188985,
  'Semantic Uncertainty': -1.3301614682619995},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -1.6569737137542835,
  'Token Probability': -1.8481321443875396,
  'Semantic Uncertainty': -1.689992253761078},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': -1.260951742252034,
  'Token Probability': -1.285684355165269,
  'Semantic Uncertainty': -1.3150809434566848},
 {'training_set': 'TruthfulQA',
  '

In [76]:
ece_improvement_df  = dataset_weighted_average_mean[["training_set", "test_set"]].copy()

ece_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"])
ece_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"])
ece_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"])

ece_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -0.12987008057111604,
  'Token Probability': -0.1431774521765397,
  'Semantic Uncertainty': -0.140767341283531},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.11374141464735982,
  'Token Probability': -0.160628414129709,
  'Semantic Uncertainty': -0.13781464650952496},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.06794281631905125,
  'Token Probability': -0.10867018764403688,
  'Semantic Uncertainty': -0.12934734209954046},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.18514544228066812,
  'Token Probability': -0.24685936858200025,
  'Semantic Uncertainty': -0.1981522996698688},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.03150608831011378,
  'Token Probability': -0.0990446339895491,
  'Semantic Uncertainty': -0.08498076807373822},
 {'training_set': 'Tru